In [ ]:
# 線性分類-感知器(Perceptron) 練習

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn import datasets

In [ ]:
%matplotlib inline

In [ ]:
iris = datasets.load_iris()
lf_x = pl.LazyFrame(iris["data"], schema=iris["feature_names"])
# print(lf_x.collect())
print("target_names:", iris["target_names"])
lf_y = pl.LazyFrame(iris["target"], schema=["target"])
# print(lf_y.collect())
lf_iris = pl.concat([lf_x, lf_y], how="horizontal")
lf_iris.collect()

In [ ]:
iris["target_names"]

In [ ]:
target_name = {0: "setosa", 1: "versicolor", 2: "virginica"}

In [ ]:
lf_iris = (
    lf_iris.with_columns(
        pl.col("target")
        .replace_strict(target_name, return_dtype=pl.String)
        .alias("target_name")
    )
    .filter(
        (pl.col("target_name") == "setosa") | (pl.col("target_name") == "versicolor")
    )
    .select(["sepal length (cm)", "petal length (cm)", "target_name"])
)
lf_iris.collect()

In [ ]:
target_class = {"setosa": 1, "versicolor": -1}

In [ ]:
lf_iris = lf_iris.with_columns(
    pl.col("target_name")
    .replace_strict(target_class, return_dtype=pl.Int64)
    .alias("target_class")
).drop("target_name")

In [ ]:
# lf_iris.explain()
lf_iris.collect()

In [ ]:
def sign(x):
    return 1 if x > 0 else -1

In [ ]:
data = lf_iris.collect().to_numpy()
df = lf_iris.collect()
w = np.array([0.0, 0.0, 0.0])
count = 0
iterator = 0
while True:
    error_in_epoch = 0  # 用來記錄這一個輪次（Epoch）中發生幾次錯誤
    for row in data:
        count += 1
        # 1. 提取特徵並補上 Bias 項 (x0 = 1.0)
        x, y = np.concatenate((np.array([1.0]), row[:2])), row[2]
        # 2. 檢查是否分類錯誤
        if sign(np.dot(w, x)) != y:
            print("iteraor:", iterator)
            iterator += 1
            error_in_epoch += 1
            # 繪製資料點
            sns.lmplot(
                x="sepal length (cm)",
                y="petal length (cm)",
                data=df,
                fit_reg=False,
                hue="target_class",
            )

            # 前一個Decision Boundary的法向量
            if w[1] != 0:
                x_last_decicion_boundary = np.linspace(0, w[1])
                y_last_decicion_boundary = w[2] / w[1] * x_last_decicion_boundary
                plt.plot(x_last_decicion_boundary, y_last_decicion_boundary)
            # 3. 更新權重 w = w + y * x
            print("x:", x)
            print("y:", y)
            print("w:", w)
            w += y * x
            print("x:", x)
            print("y:", y)
            print("w:", w)
            # x向量
            x_vector = np.linspace(0, x[1])
            y_vector = (x[2] / x[1]) * x_vector
            plt.plot(x_vector, y_vector, "b")
            # Decision Boundary 的方向向量
            x_decision_boundary = np.linspace(-0.5, 7)
            y_decision_boundary = (-w[1] / w[2]) * x_decision_boundary - (w[0] / w[2])
            plt.plot(x_decision_boundary, y_decision_boundary, "r")
            # Decision Boundary 的法向量
            x_decision_boundary_normal_vector = np.linspace(0, w[1])
            y_decision_boundary_normal_vector = (
                w[2] / w[1]
            ) * x_decision_boundary_normal_vector
            plt.plot(
                x_decision_boundary_normal_vector,
                y_decision_boundary_normal_vector,
                "g",
            )
            plt.xlim(-0.5, 7.5)
            plt.ylim(-2, 6)
            plt.show()

    # 4. 檢查結束條件：如果整輪 for 迴圈跑完，完全沒有任何錯誤點，代表完全分類成功！
    if error_in_epoch == 0:
        print(f"🎉 演算法收斂成功！總共更新了 {iterator} 次。最終權重 w 爲: {w}")
        break
